# Week 3: Laya ticket-routing specialization

Run all cells in order. The default is an offline workflow check. Mock results are simulated and do not measure training. See [the workflow guide](../README.md) for real training commands and interpretation.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile
from IPython.display import Markdown, display
from system_one_models.finetuning import verify_dataset, scenarios, fit_temperature
from system_one_models.decision_benchmark import create_backend, run_benchmark

MOCK = True
# Locate the week folder from either the root or notebook working directory.
WEEK = next(p / "week_03" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "week_03" / "prepare_data.py").is_file())
WORK = Path(tempfile.mkdtemp(prefix="system-one-week03-"))
DATA = WORK / "data" if MOCK else WEEK / "data" / "tickets-v1"
RUN = WORK / "run" if MOCK else WEEK / "runs" / "ticket-head-v1"
REPORTS = WORK / "reports"
print("Mode:", "MOCK — simulated answers" if MOCK else "REAL — weights, GPU, optional hosted calls")
print("Artifacts:", WORK)

def script(name, *arguments):
    subprocess.run([sys.executable, str(WEEK / name), *map(str, arguments)], check=True)


## Dataset and teacher review

Generate 3,000 balanced requests. Validation families and calibration families are separate. Manually review the training spot-check file before a real run. Never use the frozen test to choose training settings.

In [ ]:
if not DATA.exists():
    script("prepare_data.py", "--output", DATA, "--count", 3000)
splits = verify_dataset(DATA)
print({name: len(rows) for name, rows in splits.items()})
spot_checks = [json.loads(line) for line in (DATA / "spot-check.jsonl").read_text().splitlines()]
for example in spot_checks[:5]:
    print(example["state"], "=>", example["label"])


## Train, then fit temperature

Head-only CE is the first experiment. Real mode starts training only in an empty run folder. To try full tuning or RLCD, use the commands in the README after reviewing validation. Calibration uses its own split. The mock branch demonstrates scalar fitting with invented logits; it creates no checkpoint.

In [ ]:
if MOCK:
    import numpy as np
    fake_logits = np.array([[8., 0., 0.]] * 10)
    fake_labels = np.array([0] * 6 + [1] * 2 + [2] * 2)
    print("Demonstration temperature (invented logits):", fit_temperature(fake_logits, fake_labels))
else:
    if not (RUN / "best" / "rl_agent_config.json").exists():
        script("train.py", "--data", DATA, "--output", RUN)
    display(json.loads((RUN / "history.json").read_text()))
    if not (RUN / "calibrated" / "rl_agent_config.json").exists():
        script("calibrate.py", "--model", RUN / "best", "--data", DATA, "--output", RUN / "calibrated")


## Frozen-test comparison and legacy slice

Both test and legacy reports use the existing scoring harness. Mock model names use the same simulated distribution and do not imply different quality. Real checkpoints run sequentially to conserve GPU memory. Hosted evaluation is an optional CLI step requiring credentials.

In [ ]:
arguments = ["--data", DATA, "--output", REPORTS]
if not MOCK:
    arguments += ["--real", "--checkpoint", f"laya-head={RUN / 'best'}",
                  "--checkpoint", f"laya-calibrated={RUN / 'calibrated'}"]
script("evaluate.py", *arguments)
display(Markdown((REPORTS / "comparison.md").read_text()))


## Reliability diagram

Compare predicted-class probability with observed correctness. Each point is a probability bin. Sparse bins are unstable. Accuracy improvements and confidence improvements answer different product questions.

In [ ]:
from IPython.display import SVG
display(SVG(filename=str(REPORTS / "test" / "reliability.svg")))


## Review and next experiment

Record accuracy, answer errors, confidence reliability, and p50/p95 latency from the frozen test. Review validation before deciding on full training. In real mode, identify customer mistakes that a human review threshold should catch; the synthetic test is insufficient for deployment.

The stretch command in the README collects fresh mistakes only from training families and preserves every held-out split. Recalibrate each newly trained model. No improvement is established by this offline notebook.